# Módulo 2 — Estadística aplicada a datos de proceso

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> Describir, comparar y relacionar variables — y descubrir que la independencia de las observaciones no se sostiene.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## 1. Carga de la serie preparada

Partimos de la versión ya limpia y ordenada (`_LIMPIO.csv`). En un flujo real usarías la salida del Módulo 1.

In [ ]:
df = cargar_datos('datos_proceso_planta_LIMPIO.csv', parse_dates=['Fecha'])
df = df.sort_values('Fecha').set_index('Fecha')
df = df.asfreq('h')   # eje horario regular; expone huecos como NaN
df.head()

## 2. Descripción de una variable

In [ ]:
v = df['Tonelaje_tph'].dropna()
v.describe()

In [ ]:
print('media  ', round(v.mean(), 1))
print('mediana', round(v.median(), 1))
print('std    ', round(v.std(), 1))
print('CV %   ', round(100 * v.std() / v.mean(), 2))

In [ ]:
q1, q2, q3 = v.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
print('Q1, Q2, Q3 =', round(q1,1), round(q2,1), round(q3,1))
print('IQR =', round(iqr, 1))

## 3. Valores atípicos por regla IQR

In [ ]:
lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
at = v[(v < lim_inf) | (v > lim_sup)]
print('límites:', round(lim_inf,1), round(lim_sup,1))
print('n atípicos:', len(at))

In [ ]:
v.hist(bins=40); plt.title('Tonelaje'); plt.show()
plt.boxplot(v); plt.title('Tonelaje'); plt.show()

## 4. Comparación entre condiciones de operación

In [ ]:
df.groupby('Turno')['Tonelaje_tph'].agg(['mean','median','std','count'])

In [ ]:
df.groupby('Tipo_mineral')[['Ley_Cu_pct','Recuperacion_pct']].agg(['mean','std'])

In [ ]:
df.boxplot(column='Tonelaje_tph', by='Turno')
plt.suptitle(''); plt.title('Tonelaje por turno'); plt.show()

**Ojo:** una diferencia entre periodos puede deberse al *tiempo* o a un *cambio de condición* (aquí, la campaña de mineral A→B).

## 5. Relaciones entre variables

In [ ]:
num = df[['Tonelaje_tph','Ley_Cu_pct','Recuperacion_pct','Potencia_kW']]
num.corr().round(2)

In [ ]:
num.corr(method='spearman').round(2)

In [ ]:
plt.scatter(df['Ley_Cu_pct'], df['Recuperacion_pct'], s=4, alpha=0.3)
plt.xlabel('Ley Cu [%]'); plt.ylabel('Recuperación [%]'); plt.show()

In [ ]:
im = plt.imshow(num.corr(), vmin=-1, vmax=1, cmap='coolwarm')
plt.xticks(range(4), num.columns, rotation=45, ha='right')
plt.yticks(range(4), num.columns); plt.colorbar(im); plt.show()

## 6. Una prueba de hipótesis (ejemplo)

In [ ]:
from scipy import stats
dia = df.loc[df['Turno']=='Dia', 'Recuperacion_pct'].dropna()
noc = df.loc[df['Turno']=='Noche', 'Recuperacion_pct'].dropna()
t, p = stats.ttest_ind(dia, noc, equal_var=False)
d = (dia.mean() - noc.mean()) / np.sqrt((dia.var()+noc.var())/2)
print(f'diferencia de medias = {dia.mean()-noc.mean():.3f} pts')
print(f'p-value = {p:.2e}   |   d de Cohen = {d:.3f}')

Con miles de datos, casi cualquier diferencia sale 'significativa'. La pregunta real es si es **relevante para la operación**.

## 7. El experimento clave: mezclar el orden

In [ ]:
orig = df['Tonelaje_tph'].dropna()
mezcla = orig.sample(frac=1, random_state=0).reset_index(drop=True)
print('media / std originales:', round(orig.mean(),1), round(orig.std(),1))
print('media / std mezcladas :', round(mezcla.mean(),1), round(mezcla.std(),1))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6))
orig.reset_index(drop=True).plot(ax=ax[0], title='Orden real')
mezcla.plot(ax=ax[1], title='Orden mezclado (mismos valores)')
plt.tight_layout(); plt.show()

Los descriptivos son idénticos; la **estructura temporal desapareció**. Eso es lo que la estadística clásica no captura.

## Actividades sugeridas

1. Repite la comparación media/mediana para `Recuperacion_pct` por `Tipo_mineral`.
2. Calcula la matriz de correlación separada para mineral A y para mineral B. ¿Cambian las relaciones?
3. ¿El t-test entre turnos sigue siendo válido si las observaciones consecutivas están correlacionadas? Argumenta.
4. Construye dos series con igual media y std pero comportamiento temporal opuesto.

---
## Cierre

Sabemos describir y comparar, pero terminamos con una grieta: las observaciones de un proceso **tienen memoria**. El Módulo 3 introduce formalmente la estructura temporal.